# HireMind — 10 documented example cases from real validation pairs (E2), Colab CPU

Builds 10 HR-review cases for the prototype from real VALIDATION pairs, scored by the **frozen model** (M4 BERT-Tiny + Knowledge v1, seed 46) and structured by the **B1 extractor v3.1**. The 10 are chosen by a fixed rule (seeded), to show every situation HR meets:

| Cases | Situation |
|---|---|
| 1–2 | Good Fit, model right |
| 3 | Potential Fit, model right |
| 4–5 | No Fit, model right |
| 6–7 | Good Fit missed by the model (said Potential / No) |
| 8 | model said Good Fit, true No Fit |
| 9–10 | low-confidence cases (top probability < 0.5: the prototype flags "abstain / manual review") |

Outputs (saved to the **private** Drive folder `HireMind_PRIVATE_review_sheets/E2_cases`):
- `case_01.json` … `case_10.json`: open each in `hr_review_prototype.html` (Load case). **They contain CV text: keep private.**
- `E2_10_cases_index.csv`: IDs, labels, model output and requirement counts only (**no text**), safe to share; send it to be added to the submission.

In [ ]:
!pip -q install transformers==4.51.3 pandas==2.2.3 scikit-learn==1.6.1 huggingface-hub==0.30.2 openpyxl
%cd /content
!rm -rf _dl _s2 && git clone -q https://github.com/Hams770/HireMind-DL.git _dl && git -C _dl checkout -q 1f22f34462c400f884bb0a00f061dbea0bd687d5
!git clone -q https://github.com/Hams770/Senior-2.git _s2 && git -C _s2 checkout -q ec28dfc85e618a737cb192cbf959714cfee89aa5
!wget -q -O m4_s46.pt https://github.com/HIREMINDPROJECT/HireMind-Plan/raw/main/weights/a1_knowledge_M4_seed46_weights.pt
!echo "0953ce892ee0f075d37c1e6b4ff0bceb02a7932fddbca0ee5954f5e17fcb0c8e  m4_s46.pt" | sha256sum -c
!rm -rf _m5 && (test -d _plan || git clone -q --depth 1 https://github.com/HIREMINDPROJECT/HireMind-Plan.git _plan) && cp -r _plan/code/m5-distilbert-plan _m5 && cp _plan/A5_label_overlay_D1/train_labels_v1.csv _m5/phase_c2_m5/e2mods/


In [ ]:
import os, sys, numpy as np, pandas as pd, torch
os.environ["HIREMIND_MAX_LEN"] = "512"
sys.path.insert(0, "/content/_dl/src"); sys.path.insert(0, "/content/_s2/evaluation/knowledge")
from transformers import AutoTokenizer
from hiremind_dl import load_config
from hiremind_dl.data import as_dataset, download_raw, load_manifest, read_raw, split_frame, tokenize
from hiremind_dl.models import build_model
from hiremind_dl.evaluation import evaluate
from knowledge import features, knowledge_line
cfg = load_config()
raw = {"train": read_raw(download_raw(cfg, "train"))}   # the test file is never downloaded
man = load_manifest(cfg)
va = split_frame(raw, man, "validation", cfg["dataset"]["rows"]["validation"])
assert np.bincount(va.label_id).tolist() == [579, 294, 289]
vk = va.copy()
vk["job_description_text"] = [knowledge_line(features(cv, job)) + "\n" + job for cv, job in zip(va.resume_text, va.job_description_text)]
tok = AutoTokenizer.from_pretrained(cfg["student"]["name"], revision=cfg["student"]["revision"])
ds = as_dataset(tokenize(vk, tok, 512))
m = build_model("M0", cfg, 46); m.load_state_dict(torch.load("m4_s46.pt", map_location="cpu")); m.eval()
_, logits, pred = evaluate(m, ds, "cpu", 64)
prob = logits.softmax(-1).numpy(); pred = np.asarray(pred)
acc = 100 * (pred == va.label_id.to_numpy()).mean()
print(f"frozen model validation accuracy: {acc:.2f} (expected 56.97)")
assert abs(acc - 56.97) < 0.05, "predictions do not match the frozen model"

In [ ]:
import json, sys
sys.path.insert(0, "/content/_m5/phase_c2_m5/e2mods")
from structured_extractor import structure_pair
CL = ["No Fit", "Potential Fit", "Good Fit"]

def build(cv, jd, case_id, probs):  # same as E2_HR_prototype/make_case.py
    p = structure_pair(case_id, cv, jd, with_verbatim=True)
    byid = {r["record_id"]: r for r in p["records"]}
    reqs = []
    for l in p["requirement_links"]:
        jr = byid[l["requirement_id"]]
        reqs.append(dict(requirement_id=l["requirement_id"], category=l["category"], value=l["normalized_value"], requirement_type=l["requirement_type"],
                         status=l["link_status"], job_span=[jr["source_start_char"], jr["source_end_char"]],
                         cv_spans=[[byid[c]["source_start_char"], byid[c]["source_end_char"], byid[c]["evidence_status"]] for c in l["cv_record_ids"]]))
    k = int(np.argmax(probs))
    return dict(case_id=case_id, cv_text=cv, job_text=jd, requirements=reqs, advice_only=True,
                prediction=dict(label=CL[k], probabilities={CL[i]: round(float(probs[i]), 3) for i in range(3)}, abstain=bool(max(probs) < 0.5)))

y = va.label_id.to_numpy(); top = prob.max(1); rng = np.random.default_rng(20261124)
conf = top >= 0.5
groups = [("Good Fit, model right", (y == 2) & (pred == 2) & conf, 2), ("Potential Fit, model right", (y == 1) & (pred == 1) & conf, 1),
          ("No Fit, model right", (y == 0) & (pred == 0) & conf, 2), ("Good Fit missed", (y == 2) & (pred != 2) & conf, 2),
          ("False Good Fit (true No Fit)", (y == 0) & (pred == 2) & conf, 1), ("Low confidence (abstain)", ~conf, 2)]
chosen = []
for name, mask, n in groups:
    idx = np.where(mask)[0]; assert len(idx) >= n, name
    for i in rng.choice(idx, n, replace=False): chosen.append((name, int(i)))
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/HireMind_PRIVATE_review_sheets/E2_cases"; os.makedirs(D, exist_ok=True)
rows = []
for n, (situation, i) in enumerate(chosen, 1):
    case = build(str(va.resume_text.iloc[i]), str(va.job_description_text.iloc[i]), f"case_{n:02d}", prob[i])
    json.dump(case, open(f"{D}/case_{n:02d}.json", "w"), indent=1)
    st = pd.Series([r["status"] for r in case["requirements"]]).value_counts().to_dict()
    rows.append(dict(case=f"case_{n:02d}", situation=situation, row_uid=va.row_uid.iloc[i], job_group=va.job_group.iloc[i], true_label=CL[y[i]],
                     model_label=CL[pred[i]], p_no=round(float(prob[i, 0]), 3), p_potential=round(float(prob[i, 1]), 3), p_good=round(float(prob[i, 2]), 3),
                     abstain=bool(top[i] < 0.5), n_requirements=len(case["requirements"]),
                     n_required=sum(r["requirement_type"] == "REQUIRED" for r in case["requirements"]),
                     found=st.get("FOUND_EXPLICIT", 0) + st.get("FOUND_IMPLICIT", 0), not_mentioned=st.get("NOT_STATED_UNKNOWN", 0),
                     explicit_negative=st.get("FOUND_EXPLICIT_NEGATIVE", 0)))
idx = pd.DataFrame(rows); idx.to_csv(f"{D}/E2_10_cases_index.csv", index=False)
display(idx.drop(columns=["row_uid", "job_group"]))
print("saved to", D)